<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_10_0_Hotel_Cancellations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hotel Booking Cancellation — Classification Project


**Dataset:** Hotel Booking Demand (Antonio, Almeida & Nunes, 2019) — 119,390 reservations, 32 features, binary target (cancelled or not).


## The Business Problem


A hotel group loses revenue from last-minute cancellations — empty rooms that could have been sold.
They also risk overbooking: accepting more reservations than rooms, then compensating bumped guests.

**Goal:** Predict which reservations will be cancelled so the hotel can adjust overbooking limits, send reminders, or re-sell rooms.

**Cost structure:**
* **FN (unexpected cancellation):** empty room costs ~EUR 120 in lost revenue
* **FP (overbooked guest):** compensation + reputation damage costs ~EUR 40
* Ratio: FN is ~3x more expensive than FP. The optimal threshold should be lower than 0.5.


---
# 1. Load and Explore the Data



## 1.1 Download and Initial Inspection


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv'
df = pd.read_csv(url)
print('Shape:', df.shape)
df.head(3)


## 1.2 Target Distribution and the Accuracy Paradox
print("Loading data from TidyTuesday GitHub...")


In [ ]:
target_counts = df['is_canceled'].value_counts()
print('Target distribution:'); print(target_counts)
cancel_rate = df['is_canceled'].mean()
print('Cancellation rate: {:.3f}'.format(cancel_rate))
naive_baseline = max(target_counts) / len(df)
print('Naive baseline accuracy: {:.3f}'.format(naive_baseline))
print('A model that always predicts not cancelled achieves {:.1f}% accuracy.'.format(naive_baseline*100))


In [ ]:
sns.countplot(data=df, x='is_canceled')
plt.title('Target Distribution: Booking Cancellations')
plt.xlabel('Cancelled (0 = No, 1 = Yes)')
plt.show()


## 1.3 Data Preparation


### Dropping Data Leakage

`reservation_status` and `reservation_status_date` directly encode the cancellation outcome. Including them would cheat.
Also drop: `arrival_date_year`, `arrival_date_week_number`, `arrival_date_day_of_month` — these are specific date fragments unlikely to generalize.
`company` and `agent` have many missing values and high cardinality — drop them.


In [ ]:
leakage_cols = ['reservation_status', 'reservation_status_date', 'arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'company', 'agent']
df_clean = df.drop(columns=leakage_cols)
print('Original columns:', len(df.columns))
print('After dropping leakage:', df_clean.shape[1])


In [ ]:
X = df_clean.drop(columns=['is_canceled'])
y = df_clean['is_canceled']
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object','category']).columns.tolist()
Also drop: `arrival_date_year`, `arrival_date_week_number`, `arrival_date_day_of_month` — these are specific date fragments unlikely to generalize. We keep `arrival_date_month` to capture seasonality.
print('Categorical features:', len(categorical_cols))
print('Total features:', len(numeric_cols) + len(categorical_cols))


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
num_pipe = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))])
preprocessor = ColumnTransformer(transformers=[('num', num_pipe, numeric_cols), ('cat', cat_pipe, categorical_cols)])
print('Preprocessing pipeline ready.')


---
# 2. XGBoost Baseline Model



In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Train:', len(X_train), 'Test:', len(X_test))
print('Train cancellation rate: {:.3f}'.format(y_train.mean()))


In [ ]:
import xgboost as xgb
spw = (y_train == 0).sum() / (y_train == 1).sum()
print('scale_pos_weight = {:.2f}'.format(spw))
xgb_model = xgb.XGBClassifier(scale_pos_weight=spw, n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
pipe_xgb = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', xgb_model)])
pipe_xgb.fit(X_train, y_train)
print('XGBoost trained.')


In [ ]:
from sklearn.metrics import accuracy_score
y_pred = pipe_xgb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
naive = max(y_test.mean(), 1 - y_test.mean())
print('Naive baseline: {:.4f}'.format(naive))
print('XGBoost test:   {:.4f}'.format(acc))
print('Improvement:    {:.2f} pp'.format((acc - naive)*100))


## 2.1 Feature Importance


In [ ]:
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
cat_names = ohe.get_feature_names_out(categorical_cols)
all_names = list(numeric_cols) + list(cat_names)
importance = pipe_xgb.named_steps['classifier'].feature_importances_
feat_imp = pd.DataFrame({'feature': all_names, 'importance': importance}).sort_values('importance', ascending=True)
plt.figure(figsize=(10, 6))
plt.barh(feat_imp.tail(15)['feature'], feat_imp.tail(15)['importance'])
plt.xlabel('Importance (gain)'); plt.title('Top 15 Features by Importance')
plt.tight_layout(); plt.show()


---
# 3. Confusion Matrix and Basic Metrics


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Cancelled','Cancelled']).plot(cmap='Blues')
plt.title('Confusion Matrix at Default Threshold (0.5)'); plt.show()
tn, fp, fn, tp = cm.ravel()
print('True Negatives:  {:>6} (correctly predicted stay)'.format(tn))
print('False Positives: {:>6} (overbooked guests)'.format(fp))
print('False Negatives: {:>6} (unexpected cancellations!)'.format(fn))
print('True Positives:  {:>6} (correctly predicted cancellation)'.format(tp))


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
print('Precision: {:.4f}'.format(precision_score(y_test, y_pred)))
print('Recall:    {:.4f}'.format(recall_score(y_test, y_pred)))
print('F1-Score:  {:.4f}'.format(f1_score(y_test, y_pred)))
print('\nFull Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Not Cancelled','Cancelled']))


---
# 4. ROC Curve and AUC


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
y_proba = pipe_xgb.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)
plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, label='XGBoost (AUC = {:.3f})'.format(auc), linewidth=2)
plt.plot([0,1],[0,1],'k--', label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve'); plt.legend(); plt.grid(alpha=0.3); plt.show()
print('AUC = {:.3f}'.format(auc))


In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score
precision, recall, _ = precision_recall_curve(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
plt.figure(figsize=(8,6))
plt.plot(recall, precision, label='PR AUC = {:.3f}'.format(pr_auc), linewidth=2)
plt.axhline(y=y_test.mean(), color='r', ls='--', label='Baseline = {:.3f}'.format(y_test.mean()))
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curve')
plt.legend(); plt.grid(alpha=0.3); plt.show()
print('PR AUC = {:.3f} (baseline = {:.3f})'.format(pr_auc, y_test.mean()))


---
# 5. Threshold Tuning


The default 0.5 threshold assumes equal cost. For a hotel:
* **FN (unexpected cancellation):** ~EUR 120 in lost room revenue
* **FP (overbooked guest):** ~EUR 40 in compensation + reputation cost
The 3:1 ratio means the optimal threshold is below 0.5.


## 5.1 Youden's J Statistic


In [ ]:
youden_j = tpr - fpr
best_idx = np.argmax(youden_j)
best_thresh_youden = thresholds[best_idx]
print("Youden's J optimal threshold: {:.4f}".format(best_thresh_youden))
print('TPR: {:.4f}, FPR: {:.4f}'.format(tpr[best_idx], fpr[best_idx]))
y_pred_youden = (y_proba >= best_thresh_youden).astype(int)
print('Precision: {:.4f}'.format(precision_score(y_test, y_pred_youden)))
print('Recall:    {:.4f}'.format(recall_score(y_test, y_pred_youden)))


## 5.2 Business Cost Sensitivity


In [ ]:
cost_fn = 120.0; cost_fp = 40.0
costs = []
for thresh in thresholds:
    yh = (y_proba >= thresh).astype(int)
    cm_t = confusion_matrix(y_test, yh)
    _, fp_t, fn_t, _ = cm_t.ravel()
    costs.append(fn_t * cost_fn + fp_t * cost_fp)
best_cost_idx = np.argmin(costs)
best_thresh_cost = thresholds[best_cost_idx]
print('Cost-optimal threshold: {:.4f}'.format(best_thresh_cost))
print('Minimum cost: EUR {:.0f} on {:,} test samples'.format(costs[best_cost_idx], len(y_test)))


In [ ]:
plt.figure(figsize=(10,6))
plt.plot(thresholds, costs, linewidth=2)
plt.axvline(x=0.5, color='gray', ls='--', label='Default (0.5)')
plt.axvline(x=best_thresh_youden, color='green', ls='--', label="Youden's J ({:.3f})".format(best_thresh_youden))
plt.axvline(x=best_thresh_cost, color='red', ls='--', label='Cost-opt ({:.3f})'.format(best_thresh_cost))
plt.xlabel('Threshold'); plt.ylabel('Total Cost (EUR)')
plt.title('Cost Curve: Cancellation Threshold vs. Total Cost')
plt.legend(); plt.grid(alpha=0.3); plt.show()


In [ ]:
y_pred_cost = (y_proba >= best_thresh_cost).astype(int)
print('Threshold comparison on test set:')
print('  Default (0.5):       P={:.3f} R={:.3f} F1={:.3f}'.format(precision_score(y_test, y_pred), recall_score(y_test, y_pred), f1_score(y_test, y_pred)))
print('  Youden ({:.3f}): P={:.3f} R={:.3f} F1={:.3f}'.format(best_thresh_youden, precision_score(y_test, y_pred_youden), recall_score(y_test, y_pred_youden), f1_score(y_test, y_pred_youden)))
print('  Cost-opt ({:.3f}): P={:.3f} R={:.3f} F1={:.3f}'.format(best_thresh_cost, precision_score(y_test, y_pred_cost), recall_score(y_test, y_pred_cost), f1_score(y_test, y_pred_cost)))


---
# 6. Nested Cross-Validation and Final Model


## 6.1 Nested Cross-Validation


Hyperparameters to tune: n_estimators, max_depth, learning_rate.
Nested CV ensures the evaluation score is unbiased by the tuning process.


In [ ]:
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [3, 5],
    'classifier__learning_rate': [0.05, 0.1],
}
inner_grid = GridSearchCV(pipe_xgb, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
print('Running nested CV (5 outer x 3 inner x 8 combos)...')
nested_scores = cross_val_score(inner_grid, X, y, cv=outer_cv, scoring='roc_auc', n_jobs=-1)
print('Nested CV AUC:', nested_scores)
print('Mean AUC: {:.4f} +/- {:.4f}'.format(nested_scores.mean(), nested_scores.std()))
print('This is the honest, unbiased performance estimate.')


## 6.2 Final Production Model


The final model trains on 100% of data with optimal hyperparameters.
This is the model that gets deployed.


In [ ]:
print('Training final production model on 100% of data...')
final_grid = GridSearchCV(pipe_xgb, param_grid, cv=3, scoring='roc_auc', n_jobs=-1, verbose=0, refit=True)
final_grid.fit(X, y)
print('Best params:', final_grid.best_params_)
print('Best CV AUC: {:.4f}'.format(final_grid.best_score_))
y_proba_final = final_grid.predict_proba(X)[:, 1]
print('Full-data AUC: {:.4f}'.format(roc_auc_score(y, y_proba_final)))


In [ ]:
# Re-state costs for consistency
cost_fn = 120.0; cost_fp = 40.0

fpr_f, tpr_f, thresh_f = roc_curve(y, y_proba_final)
costs_f = []
for th in thresh_f:
    yh = (y_proba_final >= th).astype(int)
    cm_t = confusion_matrix(y, yh)
    _, fp_t, fn_t, _ = cm_t.ravel()
    costs_f.append(fn_t * cost_fn + fp_t * cost_fp)
best_f = np.argmin(costs_f)
prod_thresh = thresh_f[best_f]
print('Production threshold: {:.4f}'.format(prod_thresh))
print('Expected cost: EUR {:,.0f} on {:,} reservations'.format(costs_f[best_f], len(y)))
y_pred_prod = (y_proba_final >= prod_thresh).astype(int)
cm_prod = confusion_matrix(y, y_pred_prod)
tn_p, fp_p, fn_p, tp_p = cm_prod.ravel()
print('Confusion: TN={}, FP={}, FN={}, TP={}'.format(tn_p, fp_p, fn_p, tp_p))
print('Precision: {:.4f}'.format(precision_score(y, y_pred_prod)))
print('Recall:    {:.4f}'.format(recall_score(y, y_pred_prod)))
print('F1-Score:  {:.4f}'.format(f1_score(y, y_pred_prod)))


---
# Summary


This notebook demonstrated the complete 18_1 classification workflow on the Hotel Booking dataset:

| Concept | Section |
|---|---|
| Accuracy Paradox & Naive Baseline | 1.2 |
| Data Preparation & Leakage Prevention | 1.3 |
| XGBoost with scale_pos_weight | 2 |
| Feature Importance | 2.1 |
| Confusion Matrix (TP/TN/FP/FN) | 3 |
| Precision, Recall, F1-Score | 3 |
| Classification Report (macro vs. weighted) | 3 |
| ROC Curve & AUC | 4 |
| Precision-Recall Curve | 4 |
| Youden's J Statistic | 5.1 |
| Business Cost Sensitivity | 5.2 |
| Cost Curve & Threshold Selection | 5.2 |
| Hyperparameter Grid Search | 6.1 |
| Nested Cross-Validation | 6.1 |
| Final Production Model | 6.2 |

**Business takeaways:**
* The cost-optimal threshold is below 0.5, reflecting the higher cost of an unexpected cancellation.
* Nested CV provides the honest performance estimate for real-world deployment.
* The final model can be used to adjust overbooking levels based on predicted cancellation probability.

---
## Next Steps

Binary classification is the foundation, but many real-world problems involve three or more categories. In the next module, we'll explore **Multiclass Classification** using clinical data.

<a href="../18_2_MultiClassClassification/18_1_4_x_Exercise_Two_Parts.ipynb">Continue to: Multiclass Classification →</a>